# Train Test Creator

## Install libraries

In [254]:
import os
import sys
import random
from dotenv import load_dotenv
import pandas as pd
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from logger.logger import Logger
from utils.constants import *
from utils.utils import *
from tabular_database_driver.postgre_sql_driver import PostgreSQLDriver
from dtos.tabular_database_driver_dtos.postgre_sql_connection_dto import (
    PostgreSQLConnectionDto,
)
from dtos.tabular_database_driver_dtos.tabular_database_driver_dtos import *
from ta.ta_functions import *

load_dotenv()

True

In [255]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Parameters

In [256]:
STOCK_CODE = "VIC"
LOOKBACK_WINDOW = 50
FORECAST_HORIZON = 5
STRIDE = 1
ID_COLUMN = ["date", "code"]
TARGET_COLUMN = f"adjust"

# Inclusive
TRAIN_RANGE = ("2000-01-01", "2021-12-31")
VAL_RANGE = ("2022-01-01", "2023-12-31")
TEST_RANGE = ("2024-01-01", "2026-04-30")

In [257]:
STOCK_CODE = str.lower(STOCK_CODE)
STOCK_CODE

'vic'

In [258]:
TOTAL_WINDOW = LOOKBACK_WINDOW + FORECAST_HORIZON
TOTAL_WINDOW

55

## Load data

In [259]:
my_logger = Logger(
    file_name=f"{FEATURE_SELECTION_LOG_FILE_BASE}/{STOCK_CODE}/train_test_creator.log",
)

In [260]:
my_connection_model = PostgreSQLConnectionDto(
    logger=my_logger,
    host=os.getenv("POSTGRES_HOST"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    port=os.getenv("POSTGRES_PORT"),
    database=os.getenv("GOLD_POSTGRES_DATABASE"),
)

In [261]:
my_postgresql_driver = PostgreSQLDriver(logger=my_logger)
my_postgresql_driver.connect(my_connection_model)

<DatabaseExecutionStatus.SUCCESS: 'success'>

In [262]:
stock_df = my_postgresql_driver.select(
    schema_name=Schema.ENTERPRISE.value,
    table_name=f"unified_{STOCK_CODE}",
    order_by=["date"],
)

# cast all string columns that look numeric → float
for col in stock_df.columns:
    if stock_df[col].dtype == object:
        converted = pd.to_numeric(stock_df[col], errors="coerce")
        if converted.notna().sum() / len(stock_df) >= 0.9:
            stock_df[col] = converted

stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,date_is_year_end,date_month_sin,date_month_cos,date_dow_sin,date_dow_cos,date_quarter_sin,date_quarter_cos,date_doy_sin,date_doy_cos,date_unix_ts
0,VIC,2007-09-19,125.0,2.43,0.0,307840,38.48,125.0,125.0,125.0,...,False,-1.000000,-1.000000e-16,0.974928,-0.222521,-1.000000e+00,-1.000000e-16,-0.979614,-0.200891,1190160000
1,VIC,2007-09-20,131.0,2.55,6.0,794790,104.12,131.0,131.0,130.0,...,False,-1.000000,-1.000000e-16,0.433884,-0.900969,-1.000000e+00,-1.000000e-16,-0.982927,-0.183998,1190246400
2,VIC,2007-09-21,137.0,2.66,6.0,1224660,167.40,137.0,137.0,135.0,...,False,-1.000000,-1.000000e-16,-0.433884,-0.900969,-1.000000e+00,-1.000000e-16,-0.985948,-0.167052,1190332800
3,VIC,2007-09-24,143.0,2.78,6.0,551130,78.81,143.0,143.0,143.0,...,False,-1.000000,-1.000000e-16,0.000000,1.000000,-1.000000e+00,-1.000000e-16,-0.993257,-0.115935,1190592000
4,VIC,2007-09-25,150.0,2.92,7.0,962110,144.30,150.0,150.0,148.0,...,False,-1.000000,-1.000000e-16,0.781831,0.623490,-1.000000e+00,-1.000000e-16,-0.995105,-0.098820,1190678400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4567,VIC,2026-04-22,207.2,207.20,13.5,4713700,944.30,193.6,207.2,191.2,...,False,0.866025,-5.000000e-01,0.974928,-0.222521,1.000000e-16,-1.000000e+00,0.936881,-0.349647,1776816000
4568,VIC,2026-04-23,214.5,214.50,7.3,4258100,910.55,212.0,218.9,209.1,...,False,0.866025,-5.000000e-01,0.433884,-0.900969,1.000000e-16,-1.000000e+00,0.930724,-0.365723,1776902400
4569,VIC,2026-04-24,212.1,212.10,-2.4,4235200,909.84,215.2,221.9,208.0,...,False,0.866025,-5.000000e-01,-0.433884,-0.900969,1.000000e-16,-1.000000e+00,0.924291,-0.381689,1776988800
4570,VIC,2026-04-28,225.5,225.50,13.4,5194900,1159.30,210.0,226.9,209.8,...,False,0.866025,-5.000000e-01,0.781831,0.623490,1.000000e-16,-1.000000e+00,0.895839,-0.444378,1777334400


## Create features

In [263]:
N_LIST = [5]

In [264]:
feature_functions = [
    lambda df: add_bbands(
        df,
        n=N_LIST,
    ),
    lambda df: add_dema(
        df,
        n=N_LIST,
    ),
    lambda df: add_ema(
        df,
        n=N_LIST,
    ),
    lambda df: add_kama(
        df,
        n=N_LIST,
    ),
    lambda df: add_midpoint(
        df,
        n=N_LIST,
    ),
    lambda df: add_midprice(
        df,
        n=N_LIST,
    ),
    lambda df: add_sar(df),
    lambda df: add_sma(
        df,
        n=N_LIST,
    ),
    lambda df: add_t3(
        df,
        n=N_LIST,
    ),
    lambda df: add_tema(
        df,
        n=N_LIST,
    ),
    lambda df: add_trima(
        df,
        n=N_LIST,
    ),
    lambda df: add_wma(
        df,
        n=N_LIST,
    ),
    lambda df: add_adx(
        df,
        n=N_LIST,
    ),
    lambda df: add_aroon(
        df,
        n=N_LIST,
    ),
    lambda df: add_bop(
        df,
        n=N_LIST,
    ),
    lambda df: add_cci(
        df,
        n=N_LIST,
    ),
    lambda df: add_cmo(
        df,
        n=N_LIST,
    ),
    lambda df: add_macd(
        df,
    ),
    lambda df: add_mfi(
        df,
        n=N_LIST,
    ),
    lambda df: add_mom(
        df,
        n=N_LIST,
    ),
    lambda df: add_ppo(
        df,
    ),
    lambda df: add_roc(
        df,
        n=N_LIST,
    ),
    lambda df: add_rsi(
        df,
        n=N_LIST,
    ),
    lambda df: add_stoch(
        df,
    ),
    lambda df: add_stoch_rsi(
        df,
        n=N_LIST,
    ),
    lambda df: add_trix(
        df,
        n=N_LIST,
    ),
    lambda df: add_ultosc(
        df,
    ),
    lambda df: add_willr(
        df,
        n=N_LIST,
    ),
    lambda df: add_ad(
        df,
        n=N_LIST,
    ),
    lambda df: add_adosc(
        df,
    ),
    lambda df: add_obv(
        df,
        n=N_LIST,
    ),
    lambda df: add_ht_dcperiod(
        df,
        n=N_LIST,
    ),
    lambda df: add_ht_dcphase(
        df,
        n=N_LIST,
    ),
    lambda df: add_ht_phasor(
        df,
        n=N_LIST,
    ),
    lambda df: add_ht_sine(
        df,
        n=N_LIST,
    ),
    lambda df: add_ht_trendmode(
        df,
        n=N_LIST,
    ),
]
len(feature_functions)

36

In [265]:
def apply_features(df, funcs):
    for func in funcs:
        df = func(df)
    return df


featured_stock_df = apply_features(stock_df, feature_functions)
featured_stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,ht_trendmode_valid,ht_trendmode_signal_5,ht_trendmode_signal_5_slope,ht_trendmode_hist_5,ht_trendmode_hist_5_slope,ht_trendmode_hist_5_acceleration,ht_trendmode_hist_5_gt_0,ht_trendmode_hist_5_lt_0,ht_trendmode_hist_5_abs,ht_trendmode_5_strength
0,VIC,2007-09-19,125.0,2.43,0.0,307840,38.48,125.0,125.0,125.0,...,True,0.0,NaN,0.000000e+00,NaN,NaN,False,False,0.000000e+00,NaN
1,VIC,2007-09-20,131.0,2.55,6.0,794790,104.12,131.0,131.0,130.0,...,True,0.0,0.000000e+00,0.000000e+00,0.000000e+00,NaN,False,False,0.000000e+00,0.0
2,VIC,2007-09-21,137.0,2.66,6.0,1224660,167.40,137.0,137.0,135.0,...,True,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,False,False,0.000000e+00,0.0
3,VIC,2007-09-24,143.0,2.78,6.0,551130,78.81,143.0,143.0,143.0,...,True,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,False,False,0.000000e+00,0.0
4,VIC,2007-09-25,150.0,2.92,7.0,962110,144.30,150.0,150.0,148.0,...,True,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,False,False,0.000000e+00,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4567,VIC,2026-04-22,207.2,207.20,13.5,4713700,944.30,193.6,207.2,191.2,...,True,1.0,1.519933e-08,3.039865e-08,-1.519933e-08,7.599663e-09,True,False,3.039865e-08,0.0
4568,VIC,2026-04-23,214.5,214.50,7.3,4258100,910.55,212.0,218.9,209.1,...,True,1.0,1.013288e-08,2.026577e-08,-1.013288e-08,5.066442e-09,True,False,2.026577e-08,0.0
4569,VIC,2026-04-24,212.1,212.10,-2.4,4235200,909.84,215.2,221.9,208.0,...,True,1.0,6.755256e-09,1.351051e-08,-6.755256e-09,3.377628e-09,True,False,1.351051e-08,0.0
4570,VIC,2026-04-28,225.5,225.50,13.4,5194900,1159.30,210.0,226.9,209.8,...,True,1.0,4.503504e-09,9.007008e-09,-4.503504e-09,2.251752e-09,True,False,9.007008e-09,0.0


In [266]:
# Drop rows with missing values
featured_stock_df = featured_stock_df.dropna()

## Split Train Val Test

In [267]:
train_featured_stock_df = featured_stock_df[
    (featured_stock_df["date"] >= TRAIN_RANGE[0])
    & (featured_stock_df["date"] <= TRAIN_RANGE[1])
]
train_featured_stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,ht_trendmode_valid,ht_trendmode_signal_5,ht_trendmode_signal_5_slope,ht_trendmode_hist_5,ht_trendmode_hist_5_slope,ht_trendmode_hist_5_acceleration,ht_trendmode_hist_5_gt_0,ht_trendmode_hist_5_lt_0,ht_trendmode_hist_5_abs,ht_trendmode_5_strength
65,VIC,2007-12-19,159.0,3.09,3.0,167140,26.51,158.0,159.0,157.0,...,True,0.370370,-0.185185,-0.370370,-0.814815,-0.592593,False,True,0.370370,0.37037
66,VIC,2007-12-20,157.0,3.05,-2.0,191390,30.24,160.0,160.0,156.0,...,True,0.246914,-0.123457,-0.246914,0.123457,0.938272,False,True,0.246914,0.00000
67,VIC,2007-12-21,157.0,3.05,0.0,220330,34.56,156.0,157.0,156.0,...,True,0.164609,-0.082305,-0.164609,0.082305,-0.041152,False,True,0.164609,0.00000
68,VIC,2007-12-24,157.0,3.05,0.0,155160,24.36,157.0,158.0,156.0,...,True,0.109739,-0.054870,-0.109739,0.054870,-0.027435,False,True,0.109739,0.00000
70,VIC,2007-12-26,156.0,3.03,1.0,141200,21.88,155.0,156.0,154.0,...,True,0.604329,0.197836,0.395671,-0.197836,-0.901082,True,False,0.395671,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3510,VIC,2021-12-24,96.5,48.25,0.5,1415500,136.06,96.5,96.9,95.1,...,True,0.860549,0.069725,0.139451,-0.069725,0.034863,True,False,0.139451,0.00000
3511,VIC,2021-12-27,99.0,49.50,2.5,1907500,186.16,97.0,99.0,96.5,...,True,0.907033,0.046484,0.092967,-0.046484,0.023242,True,False,0.092967,0.00000
3512,VIC,2021-12-28,98.4,49.20,-0.6,1737300,169.92,99.1,99.3,96.5,...,True,0.938022,0.030989,0.061978,-0.030989,0.015495,True,False,0.061978,0.00000
3513,VIC,2021-12-29,95.5,47.75,-2.9,2291900,220.49,98.0,98.0,95.2,...,True,0.958681,0.020659,0.041319,-0.020659,0.010330,True,False,0.041319,0.00000


In [268]:
val_featured_stock_df = featured_stock_df[
    (featured_stock_df["date"] >= VAL_RANGE[0])
    & (featured_stock_df["date"] <= VAL_RANGE[1])
]
val_featured_stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,ht_trendmode_valid,ht_trendmode_signal_5,ht_trendmode_signal_5_slope,ht_trendmode_hist_5,ht_trendmode_hist_5_slope,ht_trendmode_hist_5_acceleration,ht_trendmode_hist_5_gt_0,ht_trendmode_hist_5_lt_0,ht_trendmode_hist_5_abs,ht_trendmode_5_strength
3515,VIC,2022-01-04,101.00,50.50,5.90,3071100,303.10,96.00,101.50,95.70,...,True,0.981636,0.009182,0.018364,-0.009182,0.004591,True,False,0.018364,0.000000
3516,VIC,2022-01-05,100.00,50.00,-1.00,3396500,342.98,100.80,102.20,99.50,...,True,0.987757,0.006121,0.012243,-0.006121,0.003061,True,False,0.012243,0.000000
3517,VIC,2022-01-06,104.50,52.25,4.50,5061400,531.06,101.00,106.40,100.50,...,True,0.658505,-0.329252,-0.658505,-0.670748,-0.664626,False,True,0.658505,0.658505
3518,VIC,2022-01-07,102.20,51.10,-2.30,3108800,321.55,106.40,106.40,102.20,...,True,0.772337,0.113832,0.227663,0.886168,1.556916,True,False,0.227663,0.227663
3521,VIC,2022-01-12,100.80,50.40,-0.20,2338500,233.63,101.50,101.50,98.00,...,True,0.228840,-0.114420,-0.228840,0.114420,-0.057210,False,True,0.228840,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4001,VIC,2023-12-21,43.20,21.60,-0.05,1890500,81.36,43.00,43.30,42.80,...,True,0.979030,0.010485,0.020970,-0.010485,0.005242,True,False,0.020970,0.000000
4002,VIC,2023-12-22,43.15,21.58,-0.05,2001100,86.04,43.20,43.35,42.75,...,True,0.986020,0.006990,0.013980,-0.006990,0.003495,True,False,0.013980,0.000000
4003,VIC,2023-12-25,43.40,21.70,0.25,1977500,85.71,43.10,43.55,43.00,...,True,0.990680,0.004660,0.009320,-0.004660,0.002330,True,False,0.009320,0.000000
4005,VIC,2023-12-27,43.60,21.80,0.05,1848500,80.88,43.65,43.95,43.60,...,True,0.662524,-0.331262,-0.662524,-0.668738,-0.665631,False,True,0.662524,0.662524


In [269]:
test_featured_stock_df = featured_stock_df[
    (featured_stock_df["date"] >= TEST_RANGE[0])
    & (featured_stock_df["date"] <= TEST_RANGE[1])
]
test_featured_stock_df

,code,date,close,adjust,change,matching_volume,matching_value,open,high,low,...,ht_trendmode_valid,ht_trendmode_signal_5,ht_trendmode_signal_5_slope,ht_trendmode_hist_5,ht_trendmode_hist_5_slope,ht_trendmode_hist_5_acceleration,ht_trendmode_hist_5_gt_0,ht_trendmode_hist_5_lt_0,ht_trendmode_hist_5_abs,ht_trendmode_5_strength
4010,VIC,2024-01-04,44.15,22.08,0.00,2337800,103.19,44.15,44.40,43.8,...,True,0.334160,-1.670798e-01,-3.341596e-01,1.670798e-01,9.164601e-01,False,True,3.341596e-01,0.0
4011,VIC,2024-01-05,44.10,22.05,-0.05,1481600,65.23,44.15,44.20,43.9,...,True,0.222773,-1.113865e-01,-2.227731e-01,1.113865e-01,-5.569327e-02,False,True,2.227731e-01,0.0
4012,VIC,2024-01-08,44.35,22.18,0.25,2534400,112.60,44.45,44.75,44.1,...,True,0.148515,-7.425769e-02,-1.485154e-01,7.425769e-02,-3.712884e-02,False,True,1.485154e-01,0.0
4013,VIC,2024-01-09,43.90,21.95,-0.45,1604800,70.73,44.30,44.40,43.9,...,True,0.099010,-4.950513e-02,-9.901025e-02,4.950513e-02,-2.475256e-02,False,True,9.901025e-02,0.0
4014,VIC,2024-01-10,43.60,21.80,-0.30,2675700,116.78,43.90,44.05,43.2,...,True,0.066007,-3.300342e-02,-6.600683e-02,3.300342e-02,-1.650171e-02,False,True,6.600683e-02,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4558,VIC,2026-04-09,149.20,149.20,-4.20,2623100,399.36,154.00,154.00,149.2,...,True,0.999999,5.843131e-07,1.168626e-06,-5.843131e-07,2.921566e-07,True,False,1.168626e-06,0.0
4564,VIC,2026-04-17,187.90,187.90,-1.40,6747900,1294.18,194.90,200.00,187.5,...,True,1.000000,5.129772e-08,1.025954e-07,-5.129772e-08,2.564886e-08,True,False,1.025954e-07,0.0
4568,VIC,2026-04-23,214.50,214.50,7.30,4258100,910.55,212.00,218.90,209.1,...,True,1.000000,1.013288e-08,2.026577e-08,-1.013288e-08,5.066442e-09,True,False,2.026577e-08,0.0
4569,VIC,2026-04-24,212.10,212.10,-2.40,4235200,909.84,215.20,221.90,208.0,...,True,1.000000,6.755256e-09,1.351051e-08,-6.755256e-09,3.377628e-09,True,False,1.351051e-08,0.0


## Standardization

In [270]:
ordinal_map = {"ht_dcphase_quadrant": [1, 2, 3, 4]}

In [271]:
def categorize_columns(df, ordinal_map: dict = None):
    """
    Auto-cast columns to suitable dtypes, then categorize into 3 lists.

    Parameters
    ----------
    df : pd.DataFrame
    ordinal_map : dict, optional
        {col_name: [ordered_categories]} for columns that should be ordinal.
        Example: {"size": ["S", "M", "L"], "priority": ["low", "med", "high"]}

    Returns
    -------
    numerical, nominal_categorical, ordinal_categorical : list of column names
    """
    ordinal_map = ordinal_map or {}
    df = df.copy()

    for col in df.columns:
        # --- 1. Try casting object/string columns ---
        if df[col].dtype == object or isinstance(df[col].dtype, pd.StringDtype):
            # Try numeric first
            converted = pd.to_numeric(df[col], errors="coerce")
            if converted.notna().sum() / len(df) >= 0.9:  # 90%+ parseable → numeric
                df[col] = converted
            else:
                # Fall through to categorical casting below
                pass

        # --- 2. Cast to ordinal categorical ---
        if col in ordinal_map:
            df[col] = pd.Categorical(df[col], categories=ordinal_map[col], ordered=True)

        # --- 3. Cast remaining object/string → nominal categorical ---
        elif df[col].dtype == object or isinstance(df[col].dtype, pd.StringDtype):
            df[col] = pd.Categorical(df[col])

    # --- 4. Categorize ---
    numerical, nominal_categorical, ordinal_categorical = [], [], []

    for col in df.columns:
        dtype = df[col].dtype
        if pd.api.types.is_numeric_dtype(dtype):
            numerical.append(col)
        elif isinstance(dtype, pd.CategoricalDtype):
            if dtype.ordered:
                ordinal_categorical.append(col)
            else:
                nominal_categorical.append(col)

    return numerical, nominal_categorical, ordinal_categorical

In [272]:
numerical, nominal, ordinal = categorize_columns(train_featured_stock_df, ordinal_map)
numerical, nominal, ordinal

(['close',
  'adjust',
  'change',
  'matching_volume',
  'matching_value',
  'open',
  'high',
  'low',
  'percent_change',
  'number_of_buy_orders',
  'buy_volume',
  'average_volume_per_buy_order',
  'number_of_sell_orders',
  'sell_volume',
  'average_volume_per_sell_order',
  'net_volume',
  'date_year',
  'date_month',
  'date_day',
  'date_week',
  'date_day_of_week',
  'date_day_of_year',
  'date_quarter',
  'date_day_of_quarter',
  'date_days_to_quarter_end',
  'date_quarter_progress',
  'date_days_in_month',
  'date_days_to_month_end',
  'date_week_of_month',
  'date_days_in_year',
  'date_days_to_year_end',
  'date_year_progress',
  'date_is_leap_year',
  'date_is_month_start',
  'date_is_month_end',
  'date_is_quarter_start',
  'date_is_quarter_end',
  'date_is_year_end',
  'date_month_sin',
  'date_month_cos',
  'date_dow_sin',
  'date_dow_cos',
  'date_quarter_sin',
  'date_quarter_cos',
  'date_doy_sin',
  'date_doy_cos',
  'date_unix_ts',
  'close_bb_5_upper',
  'close_

In [273]:
numerical, nominal, ordinal = categorize_columns(train_featured_stock_df, ordinal_map)

numerical = [c for c in numerical if c not in ID_COLUMN and c != TARGET_COLUMN]
nominal = [c for c in nominal if c not in ID_COLUMN and c != TARGET_COLUMN]
ordinal = [c for c in ordinal if c not in ID_COLUMN and c != TARGET_COLUMN]

ordinal_categories = [ordinal_map[col] for col in ordinal]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical),
        ("nom", OneHotEncoder(handle_unknown="ignore", sparse_output=False), nominal),
        ("ord", OrdinalEncoder(categories=ordinal_categories), ordinal),
    ],
    remainder="drop",
)

train_featured_scaled_stock_tensor = preprocessor.fit_transform(
    train_featured_stock_df
)  # fit+transform
val_featured_scaled_stock_tensor = preprocessor.transform(
    val_featured_stock_df
)  # transform only
test_featured_scaled_stock_tensor = preprocessor.transform(
    test_featured_stock_df
)  # transform only

In [274]:
display(train_featured_scaled_stock_tensor)
train_featured_scaled_stock_tensor.shape

array([[ 2.91142064,  1.50658599, -0.53692116, ...,  0.        ,
         1.        ,  3.        ],
       [ 2.83726076, -0.98906731, -0.51209818, ...,  0.        ,
         1.        ,  3.        ],
       [ 2.83726076,  0.00919401, -0.48247438, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [ 0.66437633, -0.29028439,  1.07033889, ...,  0.        ,
         1.        ,  3.        ],
       [ 0.55684451, -1.4382849 ,  1.63804308, ...,  0.        ,
         1.        ,  3.        ],
       [ 0.53830454, -0.24037132,  0.99366914, ...,  0.        ,
         1.        ,  0.        ]], shape=(2614, 600))

(2614, 600)

In [275]:
display(val_featured_scaled_stock_tensor)
val_featured_scaled_stock_tensor.shape

array([[ 0.76078417,  2.9540649 ,  2.43565417, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.72370424, -0.48993665,  2.76874279, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.89056396,  2.25528197,  4.47298137, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-1.37502032,  0.13397667,  1.31621438, ...,  0.        ,
         1.        ,  2.        ],
       [-1.36760433,  0.03415054,  1.18416634, ...,  0.        ,
         1.        ,  2.        ],
       [-1.33608638,  0.43345507,  3.45887292, ...,  0.        ,
         1.        ,  2.        ]], shape=(386, 600))

(386, 600)

In [276]:
display(test_featured_scaled_stock_tensor)
test_featured_scaled_stock_tensor.shape

array([[-1.34721036,  0.00919401,  1.68502762, ...,  0.        ,
         1.        ,  1.        ],
       [-1.34906436, -0.01576252,  0.80859716, ...,  0.        ,
         1.        ,  1.        ],
       [-1.33979437,  0.13397667,  1.88627292, ...,  0.        ,
         1.        ,  1.        ],
       ...,
       [ 4.96935726,  3.65284782,  3.65070084, ...,  0.        ,
         0.        ,  1.        ],
       [ 4.8803654 , -1.18871957,  3.62725976, ...,  0.        ,
         0.        ,  1.        ],
       [ 4.95081729, -5.73080857,  5.21490709, ...,  0.        ,
         0.        ,  1.        ]], shape=(387, 600))

(387, 600)

## Roll windows

In [277]:
TOTAL_WINDOW = LOOKBACK_WINDOW + FORECAST_HORIZON
TOTAL_WINDOW

55

In [278]:
def make_windows(
    X_scaled, source_df, target_col, lookback_window, forecast_horizon, stride=1
):
    prices = source_df[target_col].values
    dates = source_df["date"].values

    total_window = lookback_window + forecast_horizon
    X_list, y_list, dates_list = [], [], []

    for i in range(0, len(X_scaled) - total_window + 1, stride):
        today_idx = i + lookback_window - 1
        future_idx = i + lookback_window + forecast_horizon - 1

        X_list.append(X_scaled[i : i + lookback_window])
        y_list.append(prices[future_idx] / prices[today_idx] - 1)
        dates_list.append(dates[today_idx])

    X = np.array(X_list)
    y = np.array(y_list)
    dates = np.array(dates_list)

    print(f"X: {X.shape} | y: {y.shape} | dates: {dates.shape}")
    return X, y, dates

In [279]:
X_train_tensor, y_train_tensor, dates_train = make_windows(
    train_featured_scaled_stock_tensor,
    train_featured_stock_df,
    TARGET_COLUMN,
    LOOKBACK_WINDOW,
    FORECAST_HORIZON,
    STRIDE,
)
X_val_tensor, y_val_tensor, dates_val = make_windows(
    val_featured_scaled_stock_tensor,
    val_featured_stock_df,
    TARGET_COLUMN,
    LOOKBACK_WINDOW,
    FORECAST_HORIZON,
    STRIDE,
)
X_test_tensor, y_test_tensor, dates_test = make_windows(
    test_featured_scaled_stock_tensor,
    test_featured_stock_df,
    TARGET_COLUMN,
    LOOKBACK_WINDOW,
    FORECAST_HORIZON,
    STRIDE,
)

X: (2560, 50, 600) | y: (2560,) | dates: (2560,)
X: (332, 50, 600) | y: (332,) | dates: (332,)
X: (333, 50, 600) | y: (333,) | dates: (333,)


In [280]:
target_scaler = StandardScaler()
y_train_tensor = target_scaler.fit_transform(y_train_tensor.reshape(-1, 1)).flatten()
y_val_tensor = target_scaler.transform(y_val_tensor.reshape(-1, 1)).flatten()
y_test_tensor = target_scaler.transform(y_test_tensor.reshape(-1, 1)).flatten()

print(
    f"y_train_tensor — mean: {y_train_tensor.mean():.4f} | std: {y_train_tensor.std():.4f}"
)
print(
    f"y_val_tensor   — mean: {y_val_tensor.mean():.4f}   | std: {y_val_tensor.std():.4f}"
)
print(
    f"y_test_tensor  — mean: {y_test_tensor.mean():.4f}  | std: {y_test_tensor.std():.4f}"
)

y_train_tensor — mean: 0.0000 | std: 1.0000
y_val_tensor   — mean: -0.2326   | std: 0.8734
y_test_tensor  — mean: 0.4646  | std: 1.8888


In [281]:
type(X_train_tensor)

numpy.ndarray

In [282]:
dates_train[:1]

array(['2008-03-18T00:00:00.000000000'], dtype='datetime64[ns]')

## Validate windows

In [283]:
number_of_sample_windows = X_train_tensor.shape[0]
print(f"Number of sample windows: {number_of_sample_windows}")

Number of sample windows: 2560


In [284]:
sample_idx = 0

if sample_idx < 0 or sample_idx > number_of_sample_windows - 1:
    raise ValueError(f"sample_idx must be between 0 and {number_of_sample_windows - 1}")

# ── raw index positions this sample corresponds to ──
today_idx = sample_idx + LOOKBACK_WINDOW - 1
future_idx = sample_idx + LOOKBACK_WINDOW + FORECAST_HORIZON - 1

# ── feature names output by the preprocessor ──
feature_names = (
    numerical
    + preprocessor.named_transformers_["nom"].get_feature_names_out(nominal).tolist()
    + ordinal
)

print("=" * 60)
print(f"SAMPLE INDEX: {sample_idx} / {number_of_sample_windows - 1}")
print("=" * 60)

print(f"\n── Input window dates ──")
print(f"  From : {train_featured_stock_df['date'].iloc[sample_idx]}")
print(f"  To   : {train_featured_stock_df['date'].iloc[today_idx]}  ← today")

print(f"\n── Target ──")
print(
    f"  Today  date               : {train_featured_stock_df['date'].iloc[today_idx]}"
)
print(
    f"  Future date               : {train_featured_stock_df['date'].iloc[future_idx]}"
)
print(
    f"  Today  price              : {train_featured_stock_df['adjust'].iloc[today_idx]}"
)
print(
    f"  Future price              : {train_featured_stock_df['adjust'].iloc[future_idx]}"
)
print(f"  y (standardized return)   : {y_train_tensor[sample_idx]:.6f}")

print(f"\n── X[0] — scaled input window — shape {X_train_tensor[sample_idx].shape} ──")
pd.DataFrame(
    X_train_tensor[sample_idx],
    columns=feature_names,
    index=train_featured_stock_df["date"].iloc[sample_idx : today_idx + 1].values,
)

SAMPLE INDEX: 0 / 2559

── Input window dates ──
  From : 2007-12-19 00:00:00
  To   : 2008-03-18 00:00:00  ← today

── Target ──
  Today  date               : 2008-03-18 00:00:00
  Future date               : 2008-03-25 00:00:00
  Today  price              : 2.49
  Future price              : 2.25
  y (standardized return)   : -1.556546

── X[0] — scaled input window — shape (50, 600) ──


,close,change,matching_volume,matching_value,open,high,low,percent_change,number_of_buy_orders,buy_volume,...,date_day_name_Friday,date_day_name_Monday,date_day_name_Thursday,date_day_name_Tuesday,date_day_name_Wednesday,date_season_Autumn,date_season_Spring,date_season_Summer,date_season_Winter,ht_dcphase_quadrant
2007-12-19,2.911421,1.506586,-0.536921,-0.324029,2.877585,2.841137,2.923108,0.732545,-0.492654,-0.544745,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,3.0
2007-12-20,2.837261,-0.989067,-0.512098,-0.283503,2.951760,2.877763,2.885457,-0.502410,-0.541902,-0.533478,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0
2007-12-21,2.837261,0.009194,-0.482474,-0.236567,2.803410,2.767883,2.885457,-0.013089,-0.439888,-0.463828,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2007-12-24,2.837261,0.009194,-0.549184,-0.347388,2.840498,2.804510,2.885457,-0.013089,-0.512001,-0.562597,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2007-12-26,2.800181,0.508325,-0.563474,-0.374333,2.766323,2.731256,2.810156,0.239339,-0.492654,-0.562331,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2007-12-27,2.763101,-0.489937,-0.558059,-0.365098,2.803410,2.731256,2.847807,-0.261633,-0.542782,-0.586944,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2007-12-28,2.763101,0.009194,-0.482628,-0.242760,2.729235,2.694629,2.810156,-0.013089,-0.514640,-0.508249,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2008-01-02,2.837261,1.007455,-0.548161,-0.347171,2.766323,2.767883,2.847807,0.487884,-0.565647,-0.570145,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2008-01-03,2.726021,-1.488198,-0.593374,-0.423876,2.803410,2.731256,2.810156,-0.754838,-0.531349,-0.619530,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2008-01-04,2.651861,-0.989067,-0.639765,-0.501559,2.692148,2.621375,2.734855,-0.517944,-0.510242,-0.634052,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0


## Create metadata JSON

In [285]:
metadata = {
    "stock_code": STOCK_CODE,
    "lookback_window": LOOKBACK_WINDOW,
    "forecast_horizon": FORECAST_HORIZON,
    "stride": STRIDE,
    "train_range": [
        train_featured_stock_df.date.min().strftime('%Y-%m-%d'),
        train_featured_stock_df.date.max().strftime('%Y-%m-%d')
    ],
    "train_shape": X_train_tensor.shape,
    "val_range": [
        val_featured_stock_df.date.min().strftime('%Y-%m-%d'),
        val_featured_stock_df.date.max().strftime('%Y-%m-%d')
    ],
    "val_shape": X_val_tensor.shape,
    "test_range": [
        test_featured_stock_df.date.min().strftime('%Y-%m-%d'),
        test_featured_stock_df.date.max().strftime('%Y-%m-%d')
    ],
    "test_shape": X_test_tensor.shape,
}

metadata

{'stock_code': 'vic',
 'lookback_window': 50,
 'forecast_horizon': 5,
 'stride': 1,
 'train_range': ['2007-12-19', '2021-12-30'],
 'train_shape': (2560, 50, 600),
 'val_range': ['2022-01-04', '2023-12-28'],
 'val_shape': (332, 50, 600),
 'test_range': ['2024-01-04', '2026-04-29'],
 'test_shape': (333, 50, 600)}

## Write to folder

In [286]:
TRAIN_TEST_SET_DIR

'../../train_test_set'

In [287]:
TRAIN_TEST_SET_STOCK_CODE = (
    f"{TRAIN_TEST_SET_DIR}/{STOCK_CODE}_{LOOKBACK_WINDOW}_{FORECAST_HORIZON}_{STRIDE}"
)
os.makedirs(TRAIN_TEST_SET_STOCK_CODE, exist_ok=True)
TRAIN_TEST_SET_STOCK_CODE

'../../train_test_set/vic_50_5_1'

In [288]:
metadata_path = f"{TRAIN_TEST_SET_STOCK_CODE}/metadata.json"
print(metadata_path)
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)
    print(f"Metadata written to {metadata_path}")

../../train_test_set/vic_50_5_1/metadata.json
Metadata written to ../../train_test_set/vic_50_5_1/metadata.json


In [289]:
train_featured_stock_df.to_csv(
    f"{TRAIN_TEST_SET_STOCK_CODE}/train_featured_stock_df.csv", index=False
)
val_featured_stock_df.to_csv(
    f"{TRAIN_TEST_SET_STOCK_CODE}/val_featured_stock_df.csv", index=False
)
test_featured_stock_df.to_csv(
    f"{TRAIN_TEST_SET_STOCK_CODE}/test_featured_stock_df.csv", index=False
)

np.save(
    f"{TRAIN_TEST_SET_STOCK_CODE}/train_featured_scaled_stock_tensor.npy",
    train_featured_scaled_stock_tensor,
)
np.save(
    f"{TRAIN_TEST_SET_STOCK_CODE}/val_featured_scaled_stock_tensor.npy",
    val_featured_scaled_stock_tensor,
)
np.save(
    f"{TRAIN_TEST_SET_STOCK_CODE}/test_featured_scaled_stock_tensor.npy",
    test_featured_scaled_stock_tensor,
)

np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/X_train_tensor.npy", X_train_tensor)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/X_val_tensor.npy", X_val_tensor)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/X_test_tensor.npy", X_test_tensor)

np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/y_train_tensor.npy", y_train_tensor)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/y_val_tensor.npy", y_val_tensor)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/y_test_tensor.npy", y_test_tensor)

np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/dates_train.npy", dates_train)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/dates_val.npy", dates_val)
np.save(f"{TRAIN_TEST_SET_STOCK_CODE}/dates_test.npy", dates_test)

OSError: [Errno 22] Invalid argument: '../../train_test_set/vic_50_5_1/train_featured_stock_df.csv'